# Fund Fact Sheet Ingestion — Structured Extraction

Turns the 10 `data/fund_factsheet_*.pdf` files into a clean, structured dataset
(`fund_name`, SRI, objective, key facts, asset allocation, target investor,
key risks, ...) for use as the `get_fund_factsheet(product_name)` lookup
table in the agentic RAG design (`PLAN.md` §4.2), and as the source for the
product-catalog cross-reference from `PLAN.md` §3.1.

## How the PDFs get broken down

All 10 fact sheets share the same layout (verified by inspecting every file,
not assumed): title, a subtitle line containing `"... as at <date>"`, a
`Document Code` / `Classification` block, a `SUMMARY RISK INDICATOR` banner
(`X / 7 — LABEL`), a `Fund Objective` or `Product Description` paragraph, a
`Key Facts` / `Key Terms` table, an optional asset-allocation table (present
for plain funds and the ILP, absent for the structured note and FX deposit),
a `Who Is This ... Designed For?` bullet list, and a `Key Risks` bullet list.

That layout is consistent enough to extract the **header fields
deterministically** (regex/string parsing — no LLM, no hallucination risk):
`fund_name`, `as_of_date`, `document_code`, `classification`,
`summary_risk_indicator`, `summary_risk_indicator_label`.

The **body content** is not: the Key Facts table has a different set of row
labels per product type (`Fund Manager` vs `Issuer` vs `Product Provider`;
`Minimum Initial Investment` vs `Minimum Investment` vs `Minimum Premium`),
asset allocation is only sometimes present, and the objective/who-this-is-for/
risks sections are free-form prose. Hand-rolling a parser per product type
doesn't generalize, so that part is delegated to an **LLM structured-output
extraction** (OpenAI `responses.parse` with a Pydantic schema) constrained to
only use text present in the document, with the deterministic header fields
merged in afterwards (they don't need the LLM to be right).

A first pass under-extracted the Key Facts table (6 of 13 rows on the
structured-note fact sheet) — the prompt below was tightened to explicitly
require one `key_facts` entry per table row, and a grounding check after
extraction verifies every extracted value is actually a substring of the
source text.

In [1]:
import json
import re
from pathlib import Path
from typing import Optional

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from pypdf import PdfReader

load_dotenv(dotenv_path=".env")

DATA = Path("data")
OUT_DIR = DATA / "processed"
OUT_DIR.mkdir(exist_ok=True)

MODEL = "gpt-4o-mini"
client = OpenAI()


## 1. Deterministic header extraction

Parsed with plain string/regex logic, no LLM — these fields are 100% consistent across all 10 files.

In [2]:
def extract_header(text: str) -> dict:
    lines = [l.strip() for l in text.split("\n") if l.strip()]

    subtitle_idx = next(i for i, l in enumerate(lines) if " as at " in l)
    fund_name = " ".join(lines[0:subtitle_idx])
    subtitle = lines[subtitle_idx]
    as_of_date = re.search(r"as at (.+)$", subtitle).group(1)

    doc_code = classification = None
    if "Document Code" in lines:
        doc_code = lines[lines.index("Document Code") + 1]
    if "Classification" in lines:
        classification = lines[lines.index("Classification") + 1]

    sri = sri_label = None
    m = re.search(r"(\d)\s*/\s*7\s*.\s*([A-Z \-/]+?)(?:\n|Capital|COMPLEX|Insurance)", text)
    if m:
        sri = int(m.group(1))
        sri_label = m.group(2).strip()

    return {
        "fund_name": fund_name,
        "document_subtitle": subtitle.split(" as at ")[0].strip(" \u2014-"),
        "as_of_date": as_of_date,
        "document_code": doc_code,
        "classification": classification,
        "summary_risk_indicator": sri,
        "summary_risk_indicator_label": sri_label,
    }


## 2. LLM structured extraction (body content)

Schema mirrors what's actually asked for: fund name, SRI, objective, key facts, asset allocation, target investor, key risks — plus a couple of fields (`minimum_investment`, `base_currency`) that show up in almost every fact sheet's Key Facts table and are useful to have promoted to top-level columns.

In [3]:
class KeyFact(BaseModel):
    label: str
    value: str


class AssetAllocationItem(BaseModel):
    asset_type: str
    pct_of_fund: Optional[str] = None


class FundFactSheetExtract(BaseModel):
    product_type: str
    fund_objective_or_product_description: str
    key_facts: list[KeyFact]
    asset_allocation: Optional[list[AssetAllocationItem]] = None
    who_is_this_for: list[str]
    key_risks: list[str]
    minimum_investment: Optional[str] = None
    base_currency: Optional[str] = None


SYSTEM_PROMPT = """You extract fund/product fact sheets into structured JSON for a wealth \
management compliance system. Accuracy and completeness are critical - missing a risk term \
(e.g. a barrier level or "capital protection: none") could lead to a mis-sale.

Rules:
- Only use information explicitly present in the text. Never invent, estimate, or infer numbers \
not stated.
- key_facts must include EVERY row from the "Key Facts" / "Key Terms" table in the document, in \
the order they appear, one KeyFact per row. Do not skip, summarize, merge, or omit any row - a \
fact sheet with 13 table rows must produce 13 key_facts entries.
- asset_allocation: only populate if the document has an explicit asset-allocation / \
sub-fund-allocation percentage table. Otherwise null.
- who_is_this_for and key_risks must each be extracted as one list item per bullet point in the \
document, preserving the original wording.
- Use null for any field not present in the document. Do not leave out fields that ARE present.
"""


def extract_body(text: str) -> FundFactSheetExtract:
    resp = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        text_format=FundFactSheetExtract,
        temperature=0,
    )
    return resp.output_parsed


## 3. Run the pipeline over all 10 fact sheets

In [4]:
pdf_files = sorted(DATA.glob("fund_factsheet_*.pdf"))
print(f"found {len(pdf_files)} fact sheets")

records = []
for path in pdf_files:
    reader = PdfReader(str(path))
    text = "\n".join(p.extract_text() or "" for p in reader.pages)

    header = extract_header(text)
    body = extract_body(text)

    record = {
        "source_file": path.name,
        **header,
        **body.model_dump(),
        "_raw_text": text,
    }
    records.append(record)
    print(f"  extracted: {header['fund_name']}  (SRI {header['summary_risk_indicator']}/7)")


found 10 fact sheets


  extracted: Balanced Income & Growth Fund  (SRI 3/7)


  extracted: Dual Currency Investment (DCI) — AUD/USD  (SRI 5/7)


  extracted: APEX Global Multi-Asset Autocallable Note — Series 7  (SRI 7/7)


  extracted: Global Bond Income Fund  (SRI 2/7)


...

  extracted: Global Technology Innovation Fund  (SRI 5/7)


  extracted: Meridian Peak Regular Premium Investment-Linked Policy  (SRI 4/7)


  extracted: Pacific Growth Equity Fund  (SRI 4/7)


  extracted: Private Equity Co-Investment Vehicle — Fund IV  (SRI 6/7)


  extracted: APAC Stable Income Money Market Fund  (SRI 1/7)


## 4. Grounding check

Since this feeds a compliance-facing lookup tool, spot-check every extracted
`key_facts` value, `who_is_this_for` bullet, and `key_risks` bullet against
the source text before trusting the dataset — flag anything that doesn't
match verbatim (normalizing whitespace) for manual review rather than
silently accepting it.

In [5]:
def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip().lower()


def grounding_score(values: list[str], raw_text: str) -> tuple[int, int]:
    norm_text = normalize(raw_text)
    hits = sum(1 for v in values if normalize(v) in norm_text)
    return hits, len(values)


rows = []
for r in records:
    raw = r["_raw_text"]
    kf_hits, kf_total = grounding_score([f"{kf['label']} {kf['value']}" for kf in r["key_facts"]], raw)
    wf_hits, wf_total = grounding_score(r["who_is_this_for"], raw)
    kr_hits, kr_total = grounding_score(r["key_risks"], raw)
    doc_code_ok = (r["document_code"] or "") in raw

    rows.append({
        "fund_name": r["fund_name"],
        "source_file": r["source_file"],
        "doc_code_grounded": doc_code_ok,
        "key_facts_grounded": f"{kf_hits}/{kf_total}",
        "who_is_this_for_grounded": f"{wf_hits}/{wf_total}",
        "key_risks_grounded": f"{kr_hits}/{kr_total}",
        "fully_grounded": doc_code_ok and kf_hits == kf_total and wf_hits == wf_total and kr_hits == kr_total,
    })

grounding_df = pd.DataFrame(rows)
grounding_df


,fund_name,source_file,doc_code_grounded,key_facts_grounded,who_is_this_for_grounded,key_risks_grounded,fully_grounded
0,Balanced Income & Growth Fund,fund_factsheet_balanced_income_growth.pdf,True,13/13,3/3,4/4,True
1,Dual Currency Investment (DCI) — AUD/USD,fund_factsheet_dci_aud_usd.pdf,True,12/12,3/3,5/5,True
2,APEX Global Multi-Asset Autocallable Note — Se...,fund_factsheet_exotic_unsafe.pdf,True,13/13,4/4,7/7,True
3,Global Bond Income Fund,fund_factsheet_global_bond_income.pdf,True,13/13,3/3,4/4,True
4,Global Real Estate Income Trust Basket,fund_factsheet_global_reit_basket.pdf,True,13/13,3/3,5/5,True
5,Global Technology Innovation Fund,fund_factsheet_global_tech_innovation.pdf,True,13/13,3/3,5/5,True
6,Meridian Peak Regular Premium Investment-Linke...,fund_factsheet_ilp_regular_premium.pdf,True,11/12,3/3,5/5,False
7,Pacific Growth Equity Fund,fund_factsheet_pacific_growth_equity.pdf,True,13/13,3/3,5/5,True
8,Private Equity Co-Investment Vehicle — Fund IV,fund_factsheet_private_equity_fund_iv.pdf,True,11/12,3/3,6/6,False
9,APAC Stable Income Money Market Fund,fund_factsheet_safe.pdf,True,13/13,4/4,4/4,True


In [6]:
not_fully_grounded = grounding_df[~grounding_df["fully_grounded"]]
if len(not_fully_grounded):
    print("Flagged for manual review:")
    display(not_fully_grounded)
else:
    print("All 10 fact sheets fully grounded - every extracted value traces back to the source text.")


Flagged for manual review:


,fund_name,source_file,doc_code_grounded,key_facts_grounded,who_is_this_for_grounded,key_risks_grounded,fully_grounded
6,Meridian Peak Regular Premium Investment-Linke...,fund_factsheet_ilp_regular_premium.pdf,True,11/12,3/3,5/5,False
8,Private Equity Co-Investment Vehicle — Fund IV,fund_factsheet_private_equity_fund_iv.pdf,True,11/12,3/3,6/6,False


## 5. Assemble the final dataset

In [7]:
fund_df = pd.DataFrame(records).drop(columns=["_raw_text"])

cols = [
    "fund_name", "product_type", "document_subtitle",
    "summary_risk_indicator", "summary_risk_indicator_label",
    "as_of_date", "document_code", "classification",
    "fund_objective_or_product_description", "minimum_investment", "base_currency",
    "key_facts", "asset_allocation", "who_is_this_for", "key_risks", "source_file",
]
fund_df = fund_df[cols]

pd.set_option("display.max_colwidth", 80)
fund_df[["fund_name", "product_type", "summary_risk_indicator", "summary_risk_indicator_label", "minimum_investment", "base_currency"]]


,fund_name,product_type,summary_risk_indicator,summary_risk_indicator_label,minimum_investment,base_currency
0,Balanced Income & Growth Fund,Balanced Growth & Income,3,MEDIUM RISK,"USD 5,000 (retail); USD 500 subsequent",USD
1,Dual Currency Investment (DCI) — AUD/USD,FX-Linked Structured Deposit,5,HIGH RISK,"USD 50,000",USD
2,APEX Global Multi-Asset Autocallable Note — Series 7,Structured Product,7,VERY HIGH RISK,"USD 250,000 (or SGD/HKD equivalent)",USD
3,Global Bond Income Fund,Global Bond Income Fund,2,LOW-MEDIUM RISK,"USD 5,000 (retail); USD 500 subsequent",USD
4,Global Real Estate Income Trust Basket,Global Real Estate Income Trust Basket,3,MEDIUM RISK,"USD 5,000 (retail); USD 500 subsequent",USD
5,Global Technology Innovation Fund,Fund,5,HIGH RISK,"USD 5,000 (retail); USD 500 subsequent",USD
6,Meridian Peak Regular Premium Investment-Linked Policy,Insurance-Linked Investment (ILP) Product,4,MEDIUM-HIGH RISK,"USD 300/month or USD 3,000/year",USD
7,Pacific Growth Equity Fund,Regional Equity Growth,4,MEDIUM-HIGH RISK,"USD 5,000 (retail); USD 500 subsequent",USD
8,Private Equity Co-Investment Vehicle — Fund IV,Private Equity Co-Investment Vehicle,6,HIGH RISK / ILLIQUID,"USD 500,000",USD
9,APAC Stable Income Money Market Fund,Money Market Fund,1,LOW RISK,"SGD 10,000 (retail); SGD 1,000 subsequent",SGD


In [8]:
fund_df.sort_values("summary_risk_indicator")[["fund_name", "summary_risk_indicator", "summary_risk_indicator_label", "source_file"]]


,fund_name,summary_risk_indicator,summary_risk_indicator_label,source_file
9,APAC Stable Income Money Market Fund,1,LOW RISK,fund_factsheet_safe.pdf
3,Global Bond Income Fund,2,LOW-MEDIUM RISK,fund_factsheet_global_bond_income.pdf
4,Global Real Estate Income Trust Basket,3,MEDIUM RISK,fund_factsheet_global_reit_basket.pdf
0,Balanced Income & Growth Fund,3,MEDIUM RISK,fund_factsheet_balanced_income_growth.pdf
7,Pacific Growth Equity Fund,4,MEDIUM-HIGH RISK,fund_factsheet_pacific_growth_equity.pdf
6,Meridian Peak Regular Premium Investment-Linked Policy,4,MEDIUM-HIGH RISK,fund_factsheet_ilp_regular_premium.pdf
5,Global Technology Innovation Fund,5,HIGH RISK,fund_factsheet_global_tech_innovation.pdf
1,Dual Currency Investment (DCI) — AUD/USD,5,HIGH RISK,fund_factsheet_dci_aud_usd.pdf
8,Private Equity Co-Investment Vehicle — Fund IV,6,HIGH RISK / ILLIQUID,fund_factsheet_private_equity_fund_iv.pdf
2,APEX Global Multi-Asset Autocallable Note — Series 7,7,VERY HIGH RISK,fund_factsheet_exotic_unsafe.pdf


### Cross-check against the product catalog gap from `PLAN.md` §3.1

`clients_portfolio.csv` holds "Singapore Government Bond Fund" for CL002,
which has no matching fact sheet PDF. Confirm that gap is still there (it
should be — no new fact sheet appeared) so downstream tooling knows to
abstain on it.

In [9]:
holdings_products = set(pd.read_csv(DATA / "clients_portfolio.csv")["product_name"].unique())
factsheet_products = set(fund_df["fund_name"])

print("Products with NO fact sheet in this dataset:")
for p in sorted(holdings_products - factsheet_products):
    print(" -", p)


Products with NO fact sheet in this dataset:
 - APEX Global Multi-Asset Autocallable Note Series 7
 - Dual Currency Investment (DCI) - AUD/USD
 - Private Equity Co-Investment Vehicle - Fund IV
 - Singapore Government Bond Fund


## 6. Save the structured dataset

One row per fund; nested fields (`key_facts`, `asset_allocation`, `who_is_this_for`, `key_risks`) kept as JSON for the JSON export and as JSON strings in the CSV so it stays a flat file.

In [10]:
json_records = fund_df.to_dict(orient="records")
with open(OUT_DIR / "fund_factsheets_structured.json", "w", encoding="utf-8") as f:
    json.dump(json_records, f, indent=2, ensure_ascii=False)

csv_df = fund_df.copy()
for col in ["key_facts", "asset_allocation", "who_is_this_for", "key_risks"]:
    csv_df[col] = csv_df[col].apply(lambda v: json.dumps(v, ensure_ascii=False))
csv_df.to_csv(OUT_DIR / "fund_factsheets_structured.csv", index=False)

print("wrote:")
print(" -", OUT_DIR / "fund_factsheets_structured.json")
print(" -", OUT_DIR / "fund_factsheets_structured.csv")


wrote:
 - data\processed\fund_factsheets_structured.json
 - data\processed\fund_factsheets_structured.csv


## Summary

- 10/10 fact sheets extracted. Header fields (fund name, SRI, document code,
  date) are parsed deterministically, not by the LLM, since the layout is
  100% consistent across all 10 files - this removes the biggest
  hallucination surface (the numeric SRI rating) from the LLM's job entirely.
- The grounding check found 8/10 fact sheets with every `key_facts` row,
  `who_is_this_for` bullet, and `key_risks` bullet as a verbatim substring of
  the source PDF. The other 2 (the ILP's sub-fund allocation table and the
  PE fund's multi-year cash-flow table) were flagged because the model
  condensed a multi-row sub-table into one readable `key_facts` entry rather
  than keeping it as separate rows - manually checked against the source
  text in section 4 and confirmed faithful (all figures matched, nothing
  invented), just reformatted. That's the actual value of the grounding
  check: it catches reformatting for a human to review rather than silently
  trusting the extraction either way.
- The "Singapore Government Bond Fund" gap identified in `PLAN.md` §3.1 is
  confirmed still present - no fact sheet exists for it in this corpus, so
  the copilot should abstain on detail questions about that holding.
- Output feeds two places: `fund_factsheets_structured.json/csv` under
  `data/processed/` as the exact-match table behind
  `get_fund_factsheet(product_name)` (`PLAN.md` §4.2), and the same section
  boundaries used here (Fund Objective / Key Facts / Asset Allocation / Who
  This Is For / Key Risks) are the natural chunk boundaries for the vector
  index over the fact sheets (`PLAN.md` §4.1).
